In [ ]:
import seaborn as sns
import torch
import numpy as np
import hashlib
import matplotlib.pyplot as plt 
import sklearn
import os
import json, glob, os
import numpy as np
import h5py
import matplotlib.pyplot as plt
from torch.nn import functional as F
from eeg_bench.config import get_config_value

In [ ]:
from scipy.ndimage import gaussian_filter1d


In [ ]:
#radraid2/spanchavati/eegfm/lightning_logs_v2/v2_lejepa_pretraining_global_v2mixer_proj/version_0/checkpoints/last.ckpt

from pathlib import Path
from eeg_bench.models.clinical.EEGLejepa_model import EMBED_CACHE_VERSION

TASK_NAME = "seizure_clinical" 
SPLIT = "train"
CKPT_PATH = " /radraid2/spanchavati/eegfm/lightning_logs_rebuttal/fullds_02sig_20k/version_0/checkpoints/last.ckpt"
ATTENTIVE_PROBE = True
USE_GLOBAL_PCA = False


# Map friendly task names to cache task names
TASK_ALIASES = {
    "sleep_staging": "sleep_stages",
}

def _norm_task(task):
    return TASK_ALIASES.get(task, task)

def _ckpt_hash(ckpt_path):
    return hashlib.md5(str(ckpt_path).encode()).hexdigest()[:12]

def find_lejepa_index(task, ckpt_path, split="train", attentive=True):
    task = _norm_task(task)
    cache_dir = Path(get_config_value("cache", ".cache")) / "lejepa_embeddings"
    ckpt_hash = _ckpt_hash(ckpt_path)
    if attentive:
        pattern = f"{task}_{ckpt_hash}_*_{split}_seq_{EMBED_CACHE_VERSION}.index.json"
    else:
        pattern = f"{task}_{ckpt_hash}_*_{split}_{EMBED_CACHE_VERSION}.index.json"
    matches = sorted(cache_dir.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No cache index found for pattern: {cache_dir / pattern}")
    return matches[0]

def find_h5(task, model_tag):
    task = _norm_task(task)
    h5_root = get_config_value("make_dataset") or os.path.join(get_config_value("data"), "make_dataset")
    pattern = os.path.join(h5_root, f"{task}_{model_tag}_*.h5")
    matches = sorted(glob.glob(pattern))
    #we need to pick the one with the highest number of samples (largest file size)
    matches = sorted(matches, key=lambda p: os.path.getsize(p), reverse=True)
    if not matches:
        raise FileNotFoundError(f"No H5 found for pattern: {pattern}")
    return matches[0]

sfreq = 250  # adjust if needed
index_path = "cache/lejepa_embeddings/binary_artifact_clinical_433c0df19392_0d0d086ed73c_train_seq_v2.index.json"#find_lejepa_index(TASK_NAME, CKPT_PATH, split=SPLIT, attentive=ATTENTIVE_PROBE)
print("Using index:", index_path)

with open(index_path, "r") as f:
    index = json.load(f)

# Locate the H5 dataset (recording cache)
h5_path = find_h5(TASK_NAME, "LeJEPAClinical")
print("Using H5:", h5_path)

# Locate the LaBraM H5 dataset (recording cache)
labram_h5_path = "binary_artifact_clinical_LaBraMModel_TUAR_16_True_243.h5"

# Find a chunk with mixed labels (has both 0 and 1)
mixed = []
for shard in index["shards"]:
    labels = np.load(shard["labels"], mmap_mode="r")
    seq_ids = shard["sequence_ids"]
    for i in range(len(labels)):
        if np.unique(labels[i]).size > 1:
            mixed.append({
                "seq_id": seq_ids[i],
                "labels": labels[i].copy(),
                "emb": np.load(shard["embeddings"], mmap_mode="r")[i].copy()
            })
            # break
    # if mixed:
    #     break


def get_eeg_from_seq_id(seq_id):
    with h5py.File(h5_path, "r") as hf:
        eeg = hf[f"/recordings/{seq_id}/data"][:]  # (C, T)
        ch_names = hf[f"/recordings/{seq_id}/channels"][:]
    return eeg, ch_names


def get_labram_eeg_from_seq_id(seq_id):
    if labram_h5_path is None:
        raise FileNotFoundError("LaBraM H5 not found. Set labram_h5_path or check pattern.")
    with h5py.File(labram_h5_path, "r") as hf:
        eeg = hf[f"/recordings/{seq_id}/data"][:]  # (C, T)
        ch_names = hf[f"/recordings/{seq_id}/channels"][:]
    return eeg, ch_names

# Plot one of the mixed chunks
def plot_chunk_with_embeddings(mixed_chunk, embeddings, sfreq=250, global_pca=None, pca_min=None, pca_max=None, title=None, ax=None, eeg_override=None, ch_names_override=None):
    seq_id = mixed_chunk["seq_id"]
    labels = mixed_chunk["labels"]

    # Do PCA to get colors for the embeddings
    embeddings = embeddings.squeeze(0)  # (T, D)
    
    #norm the embeddings
    emb_mean = np.mean(embeddings, axis=0, keepdims=True)
    emb_std = np.std(embeddings, axis=0, keepdims=True) + 1e-8
    # embeddings = (embeddings - emb_mean) / emb_std
    
    if global_pca is not None:
        emb_pca = global_pca.transform(embeddings)
        if emb_pca.shape[1] > 3:
            emb_pca = emb_pca[:, 1:4]
    else:
        pca = sklearn.decomposition.PCA(n_components=3)  # RGB
        emb_pca = pca.fit_transform(embeddings)  # (T, 3)

    # Normalize to [0, 1] for visualization
    # if pca_min is None or pca_max is None:
    #     emb_pca_min = emb_pca.min(axis=0, keepdims=True)
    #     emb_pca_max = emb_pca.max(axis=0, keepdims=True)
    # else:
    #     emb_pca_min = pca_min
    #     emb_pca_max = pca_max
    # emb_pca_norm = (emb_pca - emb_pca_min) / (emb_pca_max - emb_pca_min + 1e-8)
    # emb_pca_norm = np.clip(emb_pca_norm, 0.0, 1.0)

    p_low = np.percentile(emb_pca, 2, axis=0, keepdims=True)
    p_high = np.percentile(emb_pca, 98, axis=0, keepdims=True)
    
    # Avoid division by zero if flat signal
    range_span = p_high - p_low
    range_span[range_span == 0] = 1e-8

    emb_pca_norm = (emb_pca - p_low) / range_span
    emb_pca_norm = np.clip(emb_pca_norm, 0.0, 1.0)

    # Get EEG data
    if eeg_override is None:
        eeg, ch_names = get_eeg_from_seq_id(seq_id)
    else:
        eeg, ch_names = eeg_override, ch_names_override

    C, T = eeg.shape
    t = np.arange(T) / sfreq

    if ax is None:
        plt.figure(figsize=(12, 6))
        ax = plt.gca()

        

    # Shade PCA colors across time (align spans to embedding windows)
    bounds = np.linspace(0, T, len(emb_pca_norm) + 1).astype(int)
    for i, color in enumerate(emb_pca_norm):
        ax.axvspan(bounds[i] / sfreq, bounds[i + 1] / sfreq, color=color, alpha=0.5, linewidth=.1)

    # Plot all channels with vertical offsets
    ch_std = np.std(eeg, axis=1)
    scale = float(np.median(ch_std[np.isfinite(ch_std)])) if np.any(np.isfinite(ch_std)) else 1.0
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    offset = 4.0 * scale

    for c in range(C):
        ax.plot(t, eeg[c] + c * offset, color="black", linewidth=0.6)

    # Y-axis ticks/labels (channel names)
    names = [
        (n.decode() if isinstance(n, (bytes, np.bytes_)) else str(n))
        for n in ch_names
    ]
    ax.set_yticks(np.arange(C) * offset)
    ax.set_yticklabels(names, fontsize=8)

    ax.set_xlim(t[0], t[-1])
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Channels (offset)")

    # Labels overlay (upsample 1 Hz labels to match embedding windows)
    if len(labels) != len(emb_pca_norm):
        print("Normalizing Label Length from", len(labels), "to", len(emb_pca_norm))
        repeat = int(round(len(emb_pca_norm) / len(labels)))
        labels_plot = np.repeat(labels, repeat)[:len(emb_pca_norm)]
    else:
        labels_plot = labels

    # ax2 = ax.twinx()
    # # use the same bounds as the PCA windows so time aligns
    # label_times = bounds / sfreq
    # label_vals = np.concatenate([labels_plot, labels_plot[-1:]])
    # ax2.step(label_times, label_vals, where="post", color="red", linewidth=1.5)

    # chunk_len_s = index["meta"]["chunk_len_s"]  # expect 16
    # n_emb = embeddings.shape[0]
    # embed_dt = chunk_len_s / n_emb
    # print(len(labels), n_emb, embed_dt)  # should print 16, 160, 0.1

    # ...existing code...
    # Draw a red bar at the bottom for seizure periods
    ymin, ymax = ax.get_ylim()
    bar_y = ymin - 0.05 * (ymax - ymin)  # slightly below the lowest channel
    bar_height = 0.03 * (ymax - ymin)
    for i, val in enumerate(labels_plot):
        if val == 1:
            ax.add_patch(
                plt.Rectangle(
                    (bounds[i] / sfreq, bar_y),
                    (bounds[i + 1] - bounds[i]) / sfreq,
                    bar_height,
                    color="red",
                    alpha=0.8,
                    linewidth=0,
                    # z_order=10,
                )
            )
    # ...existing code...
    ax.set_ylim(bar_y - bar_height, ymax)

    



    # ax2.set_ylabel("Label")
    # ax2.set_ylim(-0.1, 1.1)

    if title:
        ax.set_title(title)


def plot_chunk(mixed_chunk, sfreq=250, global_pca=None, pca_min=None, pca_max=None, title=None, ax=None):
    return plot_chunk_with_embeddings(
        mixed_chunk,
        mixed_chunk["emb"],
        sfreq=sfreq,
        global_pca=global_pca,
        pca_min=pca_min,
        pca_max=pca_max,
        title=title,
        ax=ax,
    )


In [ ]:
idx = 150

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pc_mapper = PCA(n_components=3)
transformed = pc_mapper.fit_transform(mixed[idx]['emb'].squeeze())
plt.scatter(transformed[:,0], transformed[:,1], c = np.repeat(mixed[idx]['labels'], 10), cmap='coolwarm')
plt.colorbar()

In [ ]:
#check one mixed
plot_chunk(mixed[150], sfreq=sfreq, title=f"Seq ID: {mixed[0]['seq_id']}")

In [ ]:
if USE_GLOBAL_PCA:
    N_embeddings = 1000
    all_embeddings = []
    for shard in index["shards"]:
        labels = np.load(shard["labels"], mmap_mode="r")
        seq_ids = shard["sequence_ids"]
        for i in range(len(labels)):
            all_embeddings.append(np.load(shard["embeddings"], mmap_mode="r")[i].squeeze(0))  # (T, D)
            if len(all_embeddings) >= N_embeddings:
                break
        if len(all_embeddings) >= N_embeddings:
            break

    global_embeddings = np.concatenate(all_embeddings, axis=0)  # (N, D)

    # Center the embeddings
    embeddings = global_embeddings - np.mean(global_embeddings, axis=0, keepdims=True)
    print("Global embeddings shape:", global_embeddings.shape)
    global_pca = sklearn.decomposition.PCA(n_components=3)
    global_pca.fit(global_embeddings)

    # Use a global min/max for consistent colors across chunks
    all_pca = global_pca.transform(global_embeddings)
    if all_pca.shape[1] > 3:
        all_pca = all_pca[:, 1:4]
    pca_min = all_pca.min(axis=0, keepdims=True)
    pca_max = all_pca.max(axis=0, keepdims=True)
else:
    global_pca = None
    pca_min = None
    pca_max = None


In [ ]:
CHUNKS_TO_PLOT = 5
#get random indices to plot 
idx_to_plot = np.random.choice(len(mixed), size=min(CHUNKS_TO_PLOT, len(mixed)), replace=False)
for i in idx_to_plot:
    plot_chunk(mixed[i], global_pca=global_pca, pca_min=pca_min, pca_max=pca_max)

In [ ]:
from eeg_bench.models.clinical.labram_model import check_and_download_pretrained_model
from eeg_bench.models.clinical.LaBraM import utils as labram_utils
from eeg_bench.models.clinical.LaBraM import modeling_finetune  # noqa: F401
from timm.models import create_model
from scipy.signal import resample

# Load LaBraM base model
labram_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(check_and_download_pretrained_model(), map_location="cpu", weights_only = False)
new_checkpoint = {}
for k, v in checkpoint["model"].items():
    if k.startswith("student."):
        new_checkpoint[k[len("student."):]] = v

labram_model = create_model(
    "labram_base_patch200_200",
    qkv_bias=False,
    rel_pos_bias=True,
    num_classes=0,
    drop_rate=0.0,
    drop_path_rate=0.1,
    attn_drop_rate=0.0,
    drop_block_rate=None,
    use_mean_pooling=True,
    init_scale=0.001,
    use_rel_pos_bias=True,
    use_abs_pos_emb=True,
    init_values=0.1,
)
labram_model.load_state_dict(new_checkpoint, strict=False)
labram_model.to(labram_device).eval()


LABRAM_PATCH_SZ       = 200   # 1 s at 200 Hz
LABRAM_STRIDE_SAMPLES = 20    # 0.1 s stride, matches Laya's 0.1 s patches


def labram_embeddings_for_seq_id(seq_id, orig_sfreq=250, target_sfreq=200,
                                 batch_size=64):
    """Dense sliding-window LaBraM embeddings on a real time axis.

    Each window holds exactly ONE time patch, so forward_features returns
    (n_win, C*1, D) and mean(dim=1) is a pure channel mean. Ported from
    _labram_patch_embs_raw_segment in analyze_embeddings_bci.ipynb.
    """
    eeg, ch_names = get_eeg_from_seq_id(seq_id)

    if orig_sfreq != target_sfreq:
        new_T  = int(round(eeg.shape[1] * target_sfreq / orig_sfreq))
        eeg_rs = resample(eeg, new_T, axis=1)
    else:
        eeg_rs = np.asarray(eeg)
    new_T = eeg_rs.shape[1]
    if new_T < LABRAM_PATCH_SZ:
        raise ValueError(f"segment shorter than one 1 s window: {new_T} samples")

    names       = [(n.decode() if isinstance(n, (bytes, np.bytes_)) else str(n))
                   for n in ch_names]
    input_chans = labram_utils.get_input_chans([n.upper() for n in names])

    starts  = np.arange(0, new_T - LABRAM_PATCH_SZ + 1,
                        LABRAM_STRIDE_SAMPLES, dtype=int)
    windows = np.stack([eeg_rs[:, s:s + LABRAM_PATCH_SZ] for s in starts],
                       axis=0)                                   # (n_win, C, 200)

    embs = []
    with torch.no_grad():
        for b0 in range(0, len(windows), batch_size):
            batch  = windows[b0:b0 + batch_size]
            tokens = (torch.from_numpy(batch).float()
                        .reshape(batch.shape[0], batch.shape[1], 1,
                                 LABRAM_PATCH_SZ) / 100.0)
            patch_embs = labram_model.forward_features(
                tokens.to(labram_device),
                input_chans=input_chans,
                return_patch_tokens=True,
            )                                                    # (n_win, C*1, D)
            assert patch_embs.shape[1] == batch.shape[1], (
                f"expected {batch.shape[1]} channel tokens, got "
                f"{patch_embs.shape[1]}; channel count and input_chans disagree")
            embs.append(patch_embs.mean(dim=1).cpu().numpy())     # -> (n_win, D)

    emb = np.concatenate(embs, axis=0)[None]      # (1, n_win, D), token axis = TIME
    return emb, eeg_rs, ch_names


In [ ]:
sns.set_context('poster')

In [ ]:
good_idx = None
for i, mix in enumerate(mixed):
    if 'recording_0314_066' in mix["seq_id"]:
        good_idx = i
        break

In [ ]:
good_idx

In [ ]:
labram_emb.shape

In [ ]:
mixed[i]["emb"].shape

In [ ]:
labram_eeg.shape

In [ ]:
CHUNKS_TO_COMPARE = 10
idx_to_compare = np.random.choice(len(mixed), size=min(CHUNKS_TO_COMPARE, len(mixed)), replace=False)
idx_to_compare = np.append(idx_to_compare, idx)  # also include the earlier idx

for i in idx_to_compare:
    seq_id = mixed[i]["seq_id"]
    # print(seq_id)
    labram_emb, labram_eeg, labram_ch_names = labram_embeddings_for_seq_id(seq_id, orig_sfreq=sfreq, target_sfreq=200)

    fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    plot_chunk_with_embeddings(
        mixed[i],
        mixed[i]["emb"],
        sfreq=sfreq,
        global_pca=global_pca,
        pca_min=pca_min,
        pca_max=pca_max,
        title="Laya",
        ax=axes[0],
    )
    plot_chunk_with_embeddings(
        mixed[i],
        labram_emb,
        sfreq=200,
        global_pca=None,
        title="LaBraM",
        ax=axes[1],
        eeg_override=labram_eeg,
        ch_names_override=labram_ch_names,
    )
    plt.tight_layout()
    

    plt.savefig(f"figures/representative_pca_2.png", dpi=300, bbox_inches="tight")
    plt.show()
